|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Speculative decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: verify, reject, and prove nothing changed<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

Implement the verification step. Prove that it does not change the model.

You measured the speedup already. This notebook is about the other half. A
speculative decoder that is 3x faster and samples from a slightly different
distribution is not a faster model. It is a different model.

In [ ]:
### run this cell
VOCAB = 8                              # a very small vocabulary
target = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.01, 0.01])
draft  = np.array([0.30, 0.30, 0.20, 0.10, 0.05, 0.03, 0.01, 0.01])
print('target and draft disagree, on purpose')

# Exercise 1: accept or resample

In [ ]:
def verify(p_target, p_draft, drafted, uniform):
  """Modified rejection sampling. -> (token, True if verify accepts the draft).

  Accept the drafted token with probability min(1, p_target/p_draft). On a
  rejection, sample from the NORMALIZED residual max(0, p_target - p_draft).
  These two rules together give p_target exactly, for any draft distribution.
  """
  if uniform < min(1.0, p_target[drafted] / p_draft[drafted]):
    return drafted, True
  residual = np.maximum(0.0, p_target - p_draft)
  residual = residual / residual.sum()
  return int(rng.choice(len(residual), p=residual)), False

def draft_and_verify(draft_probs):
  """Draft one token from draft_probs, then verify it against target."""
  drafted = int(rng.choice(VOCAB, p=draft_probs))
  return verify(target, draft_probs, drafted, rng.random())

num_accepted = sum(draft_and_verify(draft)[1] for _ in range(10))
print(f'{num_accepted}/10 drafts accepted')

# Exercise 2: is the output distribution unchanged?

Two hundred thousand draws against a distribution you know exactly. This is
the only test that can catch a subtly wrong verifier.

In [ ]:
NUM_TRIALS = 200_000

def run_experiment(draft_probs):
  """-> (the empirical distribution of the output, the acceptance rate)."""
  counts = np.zeros(VOCAB)
  num_accepted = 0
  for _ in range(NUM_TRIALS):
    token, accepted = draft_and_verify(draft_probs)
    counts[token] += 1
    num_accepted += accepted
  return counts / NUM_TRIALS, num_accepted / NUM_TRIALS

empirical, acceptance_rate = run_experiment(draft)
print(f"{'token':>6} {'target':>8} {'sampled':>9} {'error':>8}")
for token, (expected, sampled) in enumerate(zip(target, empirical)):
  print(f'{token:>6} {expected:>8.3f} {sampled:>9.3f} {sampled-expected:>+8.4f}')
print(f'\nmax error {np.abs(empirical-target).max():.4f}, acceptance {acceptance_rate:.1%}')

# Exercise 3: now use a draft model that knows nothing

Uniform over the vocabulary. Predict both numbers before you run it.

In [ ]:
terrible = np.full(VOCAB, 1.0/VOCAB)    # a draft model that knows nothing
empirical, acceptance_rate = run_experiment(terrible)
print(f'uniform draft: acceptance {acceptance_rate:.1%}, '
      f'max distribution error {np.abs(empirical-target).max():.4f}')
print('\nslower, and still exactly the right distribution.')

### The property that makes this safe to ship

Exercise 3 is the point. A draft model that is pure noise gives you a low
acceptance rate, and therefore no speedup. The output distribution is still
exactly the target distribution, to the sampling error.

That result is unusual. Most approximations trade accuracy for speed, and you
must decide how much accuracy you give up. This one trades **speed for
speed**. A better draft is faster. A worse draft is slower. Neither one
changes what the model says.

So you can ship a draft model that you are not sure about. In the worst case
you wasted the compute. You did not quietly change every answer that your
users get.

Two details are easy to get wrong:

- **Clamp the residual at zero before you normalise it.** On a rejection, a
  token that the draft over-weights contributes nothing. It was already
  over-represented among the acceptances.
- **Use `min(1, ...)`**, and not the raw ratio. Where the target likes a token
  more than the draft does, you accept it every time. The extra mass arrives
  through the residual path.

    ./vc guide 17